# CISI Corpus Text Preprocessing Pipeline

This notebook implements a complete text preprocessing and normalization pipeline using the **CISI corpus** as the input dataset.

### Pipeline Steps Included:
1. **Dataset Loading & Parsing**: Downloading and extracting the CISI textual fields.
2. **Case Normalization**: Converting text to lowercase.
3. **Diacritics Removal**: Stripping accents and normalizing unicode characters.
4. **Jargon, Abbreviation & URL Normalization**: Cleaning non-standard textual noise and expanding short forms.
5. **Tokenization**: Segmenting sentences into individual token units.
6. **Noise & Punctuation Removal**: Eliminating numbers, punctuation signs, and non-informative symbols.
7. **Stopword Removal**: Filtering out high-frequency words with low semantic value.
8. **Stemming**: Reducing words to their crude base/root form.
9. **Lemmatization**: Resolving tokens to their dictionary form (lemma) using linguistic context.

---
## 0. Environment Setup & Dependencies
We start by installing and importing required libraries like `nltk` for NLP tasks and `unicodedata` for text normalization.

In [ ]:
import importlib
import subprocess
import sys

librerias_proyecto = {"numpy": "numpy",
             "pandas": "pandas",
             "matplotlib":"matplotlib",
             "seaborn":"seaborn",
             "openpyxl": "openpyxl",
             "natural language Toolkit": "nltk",
             "requests": "requests"
             }
             
print("====== INICIANDO VERIFICACIÓN DE ENTORNO ======\n")

# 2. Recorrer y validar cada librería de la lista
for nombre_importar, nombre_pip in librerias_proyecto.items():
    instalada = False
    
    while not instalada:
        try:
            # Intenta cargar la librería dinámicamente
            modulo = importlib.import_module(nombre_importar)
            instalada = True
            
            # Obtener la versión de forma segura
            version = getattr(modulo, "__version__", "Versión no expuesta")
            print(f"[{nombre_importar}] Lista para usar. Versión: {version}")
            
        except ImportError:
            print(f"[{nombre_importar}] No encontrada. Instalando vía pip: '{nombre_pip}'...")
            try:
                # Instala usando el entorno de Python que está ejecutando el script
                subprocess.check_call([sys.executable, "-m", "pip", "install", nombre_pip])
                print(f"[{nombre_importar}] Instalación enviada. Validando acceso...")
            except subprocess.CalledProcessError:
                print(f"[ERROR CRÍTICO] Falló la instalación de '{nombre_pip}'.")
                print("Revisa tu conexión a internet o los permisos de administrador.\n")
                break  # Rompe el ciclo while de esta librería para pasar a la siguiente

print("\n====== PROCESO DE CONFIGURACIÓN FINALIZADO ======")

In [ ]:
import re
import os
import requests
import unicodedata
from pathlib import Path

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download essential NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

## 2. Fetch and Parse the CISI Corpus

The CISI collection contains information retrieval documents. We fetch the raw `.ALL` text file and extract document IDs, titles, authors, and text abstracts.

In [ ]:
def cargar_cisi(ruta_carpeta):
    """Lee el archivo CISI.ALL desde un directorio local."""
    ruta_archivo = os.path.join(ruta_carpeta, "CISI.ALL")

    # Validar que el archivo exista en la ruta
    if not os.path.exists(ruta_archivo):
        raise FileNotFoundError(f"No se encontró CISI.ALL en: {ruta_carpeta}")

    # Leer el contenido del archivo
    with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
        contenido = f.read()

    return contenido


In [ ]:
# 1. Obtenemos la ruta de la carpeta actual donde está el notebook
carpeta_actual = Path('.').resolve()

# 2. Subimos un nivel en la estructura de carpetas (una carpeta por encima)
carpeta_superior = carpeta_actual.parent

texto_corpus = cargar_cisi(carpeta_superior + '/data')

In [ ]:
def parsear_cisi(texto_completo):
    documentos = []
    # El delimitador estándar del corpus CISI para iniciar un documento es '.I'
    bloques = texto_completo.split(".I ")

    for bloque in bloques[1:]:  # Omitir el primer elemento vacío
        lineas = bloque.split("\n")
        doc_id = lineas[0].strip()

        # Unir el resto de las líneas para procesar el texto (Título, Autor, Abstract)
        resto_texto = "\n".join(lineas[1:])
        

## 3. Designing the Preprocessing Pipeline Component-by-Component

Below, we define a modular mapping class to normalize shortcuts/jargon, remove diacritics, and isolate structural metadata from noise.
   

In [ ]:
# Mapping dictionary for standard abbreviations/jargon expansion
ABBREVIATION_MAP = {
    "i.e.": "that is",
    "e.g.": "for example",
    "etc.": "and so on",
    "viz.": "namely",
    "ir": "information retrieval",
    "dbms": "database management system",
    "info": "information"
    }

def remove_diacritics(text):
    """Removes accents/diacritics from unicode characters."""
    normalized = unicodedata.normalize('NFKD', text)
    return "".join([c for c in normalized if not unicodedata.combining(c)])

def normalize_jargon_and_urls(text):
    """Removes URLs and expands common domain abbreviations."""
    # Remove URLs
    text = re.sub(r'https?:\\/\\/\\S+|www\\.\\S+', '', text)
    
    # Standardize abbreviations
    for abbr, expansion in ABBREVIATION_MAP.items():
        # Case-insensitive replacement matching word boundaries
        pattern = re.compile(r'\\b' + re.escape(abbr) + r'\\b', re.IGNORECASE)
        text = pattern.sub(expansion, text)

    return text

## 4. Main Integrated Text Pipeline Function

This unified function passes data through every single technical requirement specified.

In [ ]:
def text_preprocessing_pipeline(text):
    """
    Full text preprocessing pipeline running requested normalizations consecutively.
    """
    # 1. Case Normalization (conversion to lowercase)
    processed_text = text.lower()
    
    # 2. Diacritics and Accents Removal
    processed_text = remove_diacritics(processed_text)
    
    # 3. Jargon, Slang, Abbreviations, and URL Normalization
    processed_text = normalize_jargon_and_urls(processed_text)
    
    # 4. Tokenization
    tokens = word_tokenize(processed_text)
    
    # 5. Noise and Punctuation Removal
    # Retain only purely alphabetical tokens, dropping numbers/punctuation codes
    tokens = [t for t in tokens if t.isalpha()]
    
    # 6. Stopword Removal
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    
    # 7. Stemming
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    
    # 8. Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return {
        "original": text,
        "cleaned_tokens": tokens,
        "stemmed": stemmed_tokens,
        "lemmatized": lemmatized_tokens
    }

## 5. Execution & Pipeline Testing

Let us run a rigorous evaluation on an artificial dirty sentence string followed by execution over the actual parsed CISI text block fields.
   

In [ ]:
# Validation test on a complex string showing all capabilities

test_sentence = "The DBMS framework, e.g., IR system's performance, cost $120M! Visit http://example.com. It's naïve."
results = text_preprocessing_pipeline(test_sentence)

print("--- Validation Test Sample ---")
print(f"Original:    {results['original']}")
print(f"Tokens:      {results['cleaned_tokens']}")
print(f"Stemmed:     {results['stemmed']}")
print(f"Lemmatized:  {results['lemmatized']}")

In [ ]:
# Execute pipeline on the actual Abstract text field of CISI Document ID #1

cisi_doc_1_raw = cisi_docs[1]['text']
cisi_pipeline_out = text_preprocessing_pipeline(cisi_doc_1_raw)

print("--- CISI Document #1 Preprocessing Pipeline Execution ---")
print(f"Raw Document Text:\\n{cisi_pipeline_out['original'][:350]}...\\n")
print(f"Preprocessed Lemmatized Sequence:\\n{cisi_pipeline_out['lemmatized'][:20]}...")